<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/03_preentrenar_gpt_diminuto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 03 · Preentrenar un GPT diminuto desde cero

**Objetivo:** entrenar tu propio modelo de lenguaje (¡con pesos aleatorios al principio!) usando solo tus textos.

La tarea es siempre la misma: **predecir el siguiente token**. El modelo ve un trozo de texto y
tiene que adivinar qué viene después. Al principio adivina al azar; poco a poco mejora.

- La **pérdida** (*loss*) mide cuánto se equivoca. Debe **bajar** durante el entrenamiento.
- **Entrenamiento** vs. **validación**: la validación es texto que el modelo no ve al entrenar.
  Si la pérdida de entrenamiento baja pero la de validación sube, el modelo está **memorizando** (*sobreajuste*).

Nuestro modelo tiene ~2,6 millones de parámetros. GPT-3 tiene 175 000 millones. 🙂
Inspirado en [nanoGPT](https://github.com/karpathy/nanoGPT) y [minimind](https://github.com/jingyaogong/minimind).

⏱️ Con GPU T4: unos 2–4 minutos. En CPU: usa `"max_steps": 300` en los ajustes (≈ 1,5 min, resultados más pobres).

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config, set_seed, get_device
cfg = load_config(overrides={
    "tiny_gpt": {
        # "max_steps": 300,   # ← quita el «#» para entrenar menos pasos (p. ej. en CPU)
        # "n_layer": 6,       # más capas = modelo más grande
    }
})
ajustes = cfg.tiny_gpt
set_seed(cfg.seed)
print("Dispositivo:", get_device())
print(dict(ajustes))
INICIO = "En un lugar de la Mancha"    # texto con el que el modelo empezará a escribir

## 1. Textos y tokenizador
Si ya entrenaste un tokenizador en el cuaderno 02, se carga; si no, se entrena ahora.

In [ ]:
from pathlib import Path
from dhllm.data import load_texts, join_texts
from dhllm.tokenizer import SimpleTokenizer, train_bpe

textos = load_texts(cfg.data.folders)
ruta_tok = Path(cfg.tokenizer.save_dir) / "tokenizer.json"
if ruta_tok.exists():
    tok = SimpleTokenizer.load(ruta_tok)
    print("Tokenizador cargado:", tok)
else:
    tok = train_bpe(textos, vocab_size=cfg.tokenizer.vocab_size)

## 2. Convertir el corpus en números

In [ ]:
from dhllm.pretrain import encode_corpus
train_ids, val_ids = encode_corpus(tok, join_texts(textos))

## 3. Crear el modelo (vacío)

In [ ]:
from dhllm.pretrain import build_model
modelo = build_model(tok, ajustes)

Antes de entrenar, el modelo escribe sin sentido:

In [ ]:
from dhllm.sampling import generate
print(generate(modelo, tok, INICIO, max_new_tokens=40, seed=0))

## 4. ¡Entrenar!
Verás la pérdida cada cierto número de pasos y, de vez en cuando, una muestra de texto.

In [ ]:
from dhllm.pretrain import train_tiny_gpt, plot_loss
historial = train_tiny_gpt(modelo, tok, train_ids, val_ids, ajustes, sample_prompt=INICIO)
plot_loss(historial)

## 5. ¿Cómo ha evolucionado el texto generado?
**Interpretación de la pérdida:** una pérdida de ~8,3 significa «adivinar al azar» entre 4000 tokens
(ln 4000 ≈ 8,3). Si baja a ~4, el modelo duda en promedio entre unas e⁴ ≈ 55 opciones.

In [ ]:
for paso, texto in historial["samples"]:
    print(f"--- paso {paso} ---\n{texto}\n")

## 6. Generar texto nuevo

In [ ]:
for semilla in range(3):
    print(generate(modelo, tok, INICIO, max_new_tokens=80, temperature=0.8, top_k=50, seed=semilla))
    print("-" * 60)

## 7. Guardar el modelo
Lo usaremos en el cuaderno 06. (En Colab, descarga la carpeta `outputs/` si quieres conservarlo.)

In [ ]:
from dhllm.pretrain import save_tiny_gpt
save_tiny_gpt(modelo, tok, ajustes.save_dir)

## 🖐️ Tu turno
1. Cambia `INICIO` por otra frase («Sancho dijo», «Yo por bien tengo»...).
2. Entrena con más pasos (`"max_steps": 3000`). ¿Mejora la validación o empieza a subir?
3. Haz el modelo más grande (`"n_layer": 6, "n_embd": 256`) o más pequeño (`"n_layer": 2`). Compara curvas.
4. Entrena **solo** con el Lazarillo: en los ajustes añade `"data": {"folders": ["data/sample/lazarillo_de_tormes.txt"]}` dentro de `overrides`. ¿Qué pasa con tan poco texto?
5. **Para pensar:** ¿«entiende» este modelo el Quijote? ¿Qué ha aprendido exactamente?